In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# %cd /kaggle/working

# !git clone https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
# %cd /kaggle/working/WeedsGalore_Ipr28

# !ls -lah
%cd /kaggle/working
!rm -rf WeedsGalore_Ipr28
!git clone -q https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
%cd /kaggle/working/WeedsGalore_Ipr28
!git log --oneline -1
REPO = '/kaggle/working/WeedsGalore_Ipr28'

/kaggle/working
/kaggle/working/WeedsGalore_Ipr28
e4fb269 (HEAD -> main, origin/main, origin/HEAD) Exp12 doc


In [2]:
!pip install -q "transformers==4.45.2" torchmetrics absl-py opencv-python-headless pytest scikit-learn thop
import transformers, torchmetrics, sklearn
print('transformers', transformers.__version__, '| torchmetrics', torchmetrics.__version__, '| sklearn', sklearn.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 64.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 70.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.45.2 | torchmetrics 1.9.0 | sklearn 1.6.1


In [ ]:
# !pip install -q transformers==4.45.2

In [ ]:
# !pip install -q datasets huggingface_hub

In [ ]:
# from huggingface_hub import snapshot_download

# dataset_path = snapshot_download(
#     repo_id="Voxel51/weeds-galore",
#     repo_type="dataset",
#     local_dir="/kaggle/working/weeds-galore"
# )

# print(dataset_path)

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q absl-py==1.3.0 torchmetrics==0.11.4 tensorboard==2.11.0

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q --upgrade "tensorboard>=2.20,<2.21"

In [3]:
%cd /kaggle/working

!wget -O weedsgalore-dataset.zip \
"https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip"

/kaggle/working
--2026-11-10 13:49:31--  https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip
Resolving doidata.gfz.de (doidata.gfz.de)... 139.17.228.163
Connecting to doidata.gfz.de (doidata.gfz.de)|139.17.228.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 336989804 (321M) [application/zip]
Saving to: ‘weedsgalore-dataset.zip’

weedsgalore-dataset 100%[===================>] 321.38M  30.1MB/s    in 12s     

2026-11-10 13:49:44 (27.6 MB/s) - ‘weedsgalore-dataset.zip’ saved [336989804/336989804]



In [4]:
!unzip -q /kaggle/working/weedsgalore-dataset.zip \
    -d /kaggle/working/

In [ ]:
# !find /kaggle/input/datasets/thanhthao28/exp11-ckpt -type f

In [5]:
import glob, os
# 1) Ưu tiên dataset đã Add Input (thư mục chứa các ngày 2023-05-25, ...)
hits = sorted(glob.glob('/kaggle/input/**/2023-05-25/semantics', recursive=True))
if hits:
    DS = os.path.dirname(os.path.dirname(hits[0]))
elif os.path.isdir('/kaggle/working/weedsgalore-dataset/2023-05-25'):
    DS = '/kaggle/working/weedsgalore-dataset'
else:
    # 2) Nếu chưa có: tải theo cách bạn đã dùng trước đây vào /kaggle/working/weedsgalore-dataset
    #    (trang dữ liệu: https://doidata.gfz.de/weedsgalore_e_celikkan_2024/)
    raise SystemExit('Chưa tìm thấy dataset: hãy Add Input hoặc tải về /kaggle/working/weedsgalore-dataset')
print('DS =', DS)
for d in sorted(os.listdir(DS)):
    p = os.path.join(DS, d)
    if os.path.isdir(p):
        print(d, sorted(os.listdir(p)))

DS = /kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset
2023-05-25 ['images', 'instances', 'semantics']
2023-05-30 ['images', 'instances', 'semantics']
2023-06-06 ['images', 'instances', 'semantics']
2023-06-15 ['images', 'instances', 'semantics']
splits ['test.txt', 'train.txt', 'val.txt']


In [6]:
import inspect, os, sys
REPO = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(REPO)
from transformers.models.segformer.modeling_segformer import (
    SegformerEncoder, SegformerEfficientSelfAttention, SegformerOverlapPatchEmbeddings)
src_attn = inspect.getsource(SegformerEfficientSelfAttention.forward)
src_enc = inspect.getsource(SegformerEncoder.forward)
checks = {
    'Q computed before SR (query on un-reduced hidden_states)':
        src_attn.find('self.query(hidden_states)') < src_attn.find('self.sr('),
    'SR conv called on 4D tensor then layer_norm': 'self.sr(hidden_states)' in src_attn and 'self.layer_norm(hidden_states)' in src_attn,
    'encoder loop: embedding -> blocks -> norm -> reshape': all(s in src_enc for s in ['embedding_layer(hidden_states)', 'blk(hidden_states, height, width', 'norm_layer(hidden_states)']),
}
for k, v in checks.items():
    print('OK ' if v else 'FAIL', k)
os.makedirs('results/env', exist_ok=True)
with open('results/env/hf_source_check.txt', 'w') as f:
    f.write(src_attn + '\n\n' + src_enc)

import sys
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')
assert os.path.exists(f'{REPO}/src/nets/ccmc_sr.py'), 'Repo chưa có code mới: commit + push rồi chạy lại S2'
from nets.ccmc_sr import CCMCSegFormer
m = CCMCSegFormer(prior_mode='learned_weed')
for name, mod in m.named_modules():
    if name.endswith('.sr'):
        print(name, type(mod).__name__, tuple(mod.weight.shape), 'stride', mod.stride)
assert all(checks.values()), 'API HF khác giả định -> dừng lại, gửi file results/env/hf_source_check.txt để review'

OK  Q computed before SR (query on un-reduced hidden_states)
OK  SR conv called on 4D tensor then layer_norm
OK  encoder loop: embedding -> blocks -> norm -> reshape


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

encoder.encoder.block.0.0.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.0.1.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.1.0.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.1.1.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.2.0.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)
encoder.encoder.block.2.1.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)


In [7]:
%cd /kaggle/working/WeedsGalore_Ipr28
!mkdir -p results/env
!python -m pytest -q tests/test_ccmc_sr.py 2>&1 | tee results/env/unit_tests.log | tail -n 15

/kaggle/working
......................................................                   [100%]
54 passed in 34.33s


In [10]:
#E12.0 profile without accessing labels
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

assert (PROJECT / "src/scripts/exp12_profile.py").exists(), \
    "Không tìm thấy exp12_profile.py"

for bb in ['b0', 'b1', 'b2']:
    for sz in [600, 900]:
        args = [
            sys.executable,
            'src/scripts/exp12_profile.py',
            f'--backbone={bb}',
            f'--size={sz}',
            '--batch_size=1',
            '--steps=3',
            f'--out=results/exp12/profile_{bb}_{sz}_bs1.json'
        ]

        print(f"\n===== PROFILE {bb.upper()} SIZE {sz} BS1 =====")

        p = subprocess.run(args, cwd=str(PROJECT))

        if p.returncode:
            print('PROFILE FAILED (do not assume this arm is feasible):', bb, sz)
        else:
            print('PROFILE SUCCESS:', bb, sz)


===== PROFILE B0 SIZE 600 BS1 =====
{
  "backbone": "b0",
  "size": 600,
  "batch_size": 1,
  "steps": 3,
  "mean_step_s_excluding_first": 0.10364169249999122,
  "max_vram_gb": 0.970148864,
  "params": 3662291,
  "device": "Tesla T4",
  "synthetic_only": true
}
PROFILE SUCCESS: b0 600

===== PROFILE B0 SIZE 900 BS1 =====
{
  "backbone": "b0",
  "size": 900,
  "batch_size": 1,
  "steps": 3,
  "mean_step_s_excluding_first": 0.20360930049997705,
  "max_vram_gb": 2.46470912,
  "params": 3662291,
  "device": "Tesla T4",
  "synthetic_only": true
}
PROFILE SUCCESS: b0 900

===== PROFILE B1 SIZE 600 BS1 =====
{
  "backbone": "b1",
  "size": 600,
  "batch_size": 1,
  "steps": 3,
  "mean_step_s_excluding_first": 0.12713398450006252,
  "max_vram_gb": 1.322881024,
  "params": 13562963,
  "device": "Tesla T4",
  "synthetic_only": true
}
PROFILE SUCCESS: b1 600

===== PROFILE B1 SIZE 900 BS1 =====
{
  "backbone": "b1",
  "size": 900,
  "batch_size": 1,
  "steps": 3,
  "mean_step_s_excluding_first":

In [13]:
!find /kaggle/input -maxdepth 4 -type d | head -200

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/thanhthao28
/kaggle/input/datasets/thanhthao28/exp11-ckpt
/kaggle/input/datasets/thanhthao28/exp11-ckpt/results
/kaggle/input/notebooks
/kaggle/input/notebooks/thanhthao28
/kaggle/input/notebooks/thanhthao28/exp123
/kaggle/input/notebooks/thanhthao28/exp123/.virtual_documents
/kaggle/input/notebooks/thanhthao28/exp123/WeedsGalore_Ipr28
/kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset


In [14]:
#E12.1 / E12.2 training — ONLY after freezing config and registration
import json,subprocess,sys
from pathlib import Path
PROJECT=Path('/kaggle/working/WeedsGalore_Ipr28')
sys.path.insert(0,str(PROJECT/'src'))
from exp12_config import load_json,build_cli,check_run_config
common_path=PROJECT/'configs/exp12/exp12_common_config.json'
arms_path=PROJECT/'configs/exp12/exp12_arms.json'
common=load_json(common_path);arms=load_json(arms_path)
DS = "/kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset"
REPO = "/kaggle/input/notebooks/thanhthao28/exp123/WeedsGalore_Ipr28"

CKPT = "/kaggle/input/datasets/thanhthao28/exp11-ckpt/results"
assert Path(DS).is_dir()

def train_one(arm_id,seed):
 assert seed in arms['arms'][arm_id]['seeds']
 assert seed in range(5), 'Development only. Confirmation seeds 5–9 need a locked E12.6 protocol.'
 out=PROJECT/'results'/'exp12'/arm_id/f'seed{seed}'
 out.mkdir(parents=True,exist_ok=True)
 if (out/'run_summary.json').exists():
  assert not check_run_config(out/'run_config.json',common,arms)
  print('Already done:',arm_id,seed);return
 cmd=[sys.executable,'src/train.py']+build_cli(common,arms,arm_id,seed,DS,str(out),f'{arm_id}_s{seed}',str(common_path),str(arms_path))
 subprocess.run(cmd,cwd=PROJECT,check=True)
 assert not check_run_config(out/'run_config.json',common,arms)
 assert (out/'final_ema.pth').is_file()
 print('PASS',arm_id,seed)
# After B0 smoke/test and prereg review: train_one('P_B0_LLR',0)
# Kiểm tra file cấu hình
assert common_path.is_file()
assert arms_path.is_file()

# Kiểm tra đúng thư mục dự án và dataset
assert (PROJECT / "src/train.py").is_file()
assert Path(DS).is_dir()

# Chạy thử B0 seed 0
train_one("P_B0_LLR", 0)

Exp12 configuration check passed: {'arm_id': 'P_B0_LLR', 'common_sha256': 'c28cdcdf035c87b87c8bf25ca787739dd2b34b90868aeed889db4a98010ef2a0', 'arms_sha256': '391914f037d2d5eb515342ab97f6c818b50738cce04b18283fe79e6dfef96737'}
Using cuda
Cuda current device: 0
Cuda device name: Tesla T4

MODEL STATISTICS
Total Parameters: 3,662,291
Trainable Parameters: 3,662,291
Non-trainable Parameters: 0
FLOPs (forward pass): 17,088,643,072 (~17.09G)
FLOPs per pixel: 65,188
Inference time (avg 100 runs): 16.10 ms
Throughput: 62.10 img/s

Logging to: /kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B0_LLR/seed0
Epoch: 0 iter: 1, Loss: 0.97
mIoU: 14.07%
Epoch: 0, EMA Val mIoU: 31.03%
Epoch: 0, Val mIoU: 35.57% | c0=93.08 c1=0.00 c2=13.64
Saved best_val.pth (35.57%)
Epoch: 1, EMA Val mIoU: 39.33%
Epoch: 1, Val mIoU: 43.35% | c0=87.30 c1=16.89 c2=25.87
Saved best_val.pth (43.35%)
Epoch: 2, EMA Val mIoU: 53.92%
Epoch: 2, Val mIoU: 62.46% | c0=95.61 c1=39.24 c2=52.52
Saved best_val.pth (62.46%)
Epoch: 3 it

In [18]:
from exp12_config import eval_cli

arm = "P_B0_LLR"
seed = 0

ckpt = PROJECT / "results/exp12" / arm / f"seed{seed}" / "final_ema.pth"
bins = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"
assert bins.is_file(), f"Không tìm thấy: {bins}"

assert ckpt.is_file(), f"Missing checkpoint: {ckpt}"
assert bins.is_file(), f"Missing instance_bins.json: {bins}"

cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
    arms, arm, str(ckpt), DS, "val",
    str(bins),
    str(PROJECT / "results/exp12" / arm / f"seed{seed}" / "val"),
    f"{arm}_s{seed}",
    str(arms_path),
    tta="none"
)

subprocess.run(cmd, cwd=PROJECT, check=True)

{"run_name": "P_B0_LLR_s0", "split": "val", "prior_override": "none", "prior_noise": "none", "GT_LEAKAGE": false, "instance_source": {"official": 26}, "lambdas": []}
mIoU 84.02 | bg 98.39 crop 76.59 weed 77.08 | BIoU crop 65.90 weed 57.10
  small  n= 620 recall@0.5=57.10 recall@0.3=68.55 cov>=0.5=64.19
  medium n= 567 recall@0.5=74.78 recall@0.3=79.54 cov>=0.5=93.47
  large  n= 621 recall@0.5=50.24 recall@0.3=58.78 cov>=0.5=98.39
  all    n=1808 recall@0.5=60.29 recall@0.3=68.64 cov>=0.5=85.12


CompletedProcess(args=['/usr/bin/python3', 'src/evaluate_ccmc.py', '--dataset_path=/kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset', '--split=val', '--ckpt=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B0_LLR/seed0/final_ema.pth', '--ccmc=False', '--bins_json=/kaggle/working/WeedsGalore_Ipr28/doc/exp02_results/results/exp02/instance_bins.json', '--boundary_px=3', '--run_name=P_B0_LLR_s0', '--out_prefix=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B0_LLR/seed0/val', '--arm_id=P_B0_LLR', '--exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_arms.json', '--tta=none', '--weed_bias=0.0', '--backbone=b0', '--head=bilinear', '--use_low_level_refine=True', '--llr_source=raw', '--input_scale=1.0'], returncode=0)

In [26]:
import shutil

out = "/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0"
backup = out + "_backup"

shutil.copytree(out, backup, dirs_exist_ok=True)
print("Backup:", backup)

Backup: /kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0_backup


In [27]:
shutil.rmtree(out)
print("Đã xóa:", out)

Đã xóa: /kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0


In [28]:
from exp12_config import eval_cli

arm = "P_B1_LLR"
seed = 0

# 1. Train B1 ở 900×900
train_one(arm, seed)

# 2. Đánh giá validation với TTA-D4
out = PROJECT / "results/exp12" / arm / f"seed{seed}"
ckpt = out / "final_ema.pth"
bins = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"

assert ckpt.is_file()
assert bins.is_file()

cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
    arms, arm, str(ckpt), DS, "val",
    str(bins),
    str(out / "val_d4"),
    f"{arm}_s{seed}",
    str(arms_path),
    tta="d4"
)

subprocess.run(cmd, cwd=PROJECT, check=True)

Exp12 configuration check passed: {'arm_id': 'P_B1_LLR', 'common_sha256': 'c28cdcdf035c87b87c8bf25ca787739dd2b34b90868aeed889db4a98010ef2a0', 'arms_sha256': '391914f037d2d5eb515342ab97f6c818b50738cce04b18283fe79e6dfef96737'}
Using cuda
Cuda current device: 0
Cuda device name: Tesla T4

MODEL STATISTICS
Total Parameters: 13,562,963
Trainable Parameters: 13,562,963
Non-trainable Parameters: 0
FLOPs (forward pass): 23,484,956,672 (~23.48G)
FLOPs per pixel: 89,588
Inference time (avg 100 runs): 22.28 ms
Throughput: 44.88 img/s

Logging to: /kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0
Epoch: 0 iter: 1, Loss: 1.02
mIoU: 13.69%
Epoch: 0, EMA Val mIoU: 30.72%
Epoch: 0, Val mIoU: 31.25% | c0=86.67 c1=2.38 c2=4.69
Saved best_val.pth (31.25%)
Epoch: 1, EMA Val mIoU: 49.27%
Epoch: 1, Val mIoU: 52.18% | c0=94.32 c1=22.05 c2=40.19
Saved best_val.pth (52.18%)
Epoch: 2, EMA Val mIoU: 56.04%
Epoch: 2, Val mIoU: 59.46% | c0=94.76 c1=31.54 c2=52.08
Saved best_val.pth (59.46%)
Epoch: 3 i

CompletedProcess(args=['/usr/bin/python3', 'src/evaluate_ccmc.py', '--dataset_path=/kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset', '--split=val', '--ckpt=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0/final_ema.pth', '--ccmc=False', '--bins_json=/kaggle/working/WeedsGalore_Ipr28/doc/exp02_results/results/exp02/instance_bins.json', '--boundary_px=3', '--run_name=P_B1_LLR_s0', '--out_prefix=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0/val_d4', '--arm_id=P_B1_LLR', '--exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_arms.json', '--tta=d4', '--weed_bias=0.0', '--backbone=b1', '--head=bilinear', '--use_low_level_refine=True', '--llr_source=raw', '--input_scale=1.0'], returncode=0)

In [29]:
from exp12_config import eval_cli

out = PROJECT / "results/exp12" / arm / f"seed{seed}"
ckpt = out / "final_ema.pth"
bins = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"

assert ckpt.is_file()
assert bins.is_file()

cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
    arms, arm, str(ckpt), DS, "val",
    str(bins), str(out / "val_d4"),
    f"{arm}_s{seed}", str(arms_path),
    tta="d4"
)

subprocess.run(cmd, cwd=PROJECT, check=True)

{"run_name": "P_B1_LLR_s0", "split": "val", "prior_override": "none", "prior_noise": "none", "GT_LEAKAGE": false, "instance_source": {"official": 26}, "lambdas": []}
mIoU 84.39 | bg 98.41 crop 77.27 weed 77.50 | BIoU crop 66.43 weed 57.66
  small  n= 620 recall@0.5=60.32 recall@0.3=70.16 cov>=0.5=66.77
  medium n= 567 recall@0.5=75.13 recall@0.3=79.54 cov>=0.5=94.36
  large  n= 621 recall@0.5=50.89 recall@0.3=58.62 cov>=0.5=98.87
  all    n=1808 recall@0.5=61.73 recall@0.3=69.14 cov>=0.5=86.45


CompletedProcess(args=['/usr/bin/python3', 'src/evaluate_ccmc.py', '--dataset_path=/kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset', '--split=val', '--ckpt=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0/final_ema.pth', '--ccmc=False', '--bins_json=/kaggle/working/WeedsGalore_Ipr28/doc/exp02_results/results/exp02/instance_bins.json', '--boundary_px=3', '--run_name=P_B1_LLR_s0', '--out_prefix=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B1_LLR/seed0/val_d4', '--arm_id=P_B1_LLR', '--exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_arms.json', '--tta=d4', '--weed_bias=0.0', '--backbone=b1', '--head=bilinear', '--use_low_level_refine=True', '--llr_source=raw', '--input_scale=1.0'], returncode=0)

In [31]:
#train B0/B2 seed 1–4, sau đó tự động đánh giá validation TTA-D4 bằng final_ema.pth.
from exp12_config import eval_cli
import subprocess
import sys

BINS = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"
assert BINS.is_file()

for arm in ["P_B0_LLR", "P_B2_LLR"]:
    for seed in range(1, 5):
        print(f"\n===== {arm} | SEED {seed} =====", flush=True)

        # Train; hàm train_one kiểm tra config và bỏ qua run đã hoàn thành
        train_one(arm, seed)

        # Kiểm tra checkpoint cuối
        out = PROJECT / "results/exp12" / arm / f"seed{seed}"
        ckpt = out / "final_ema.pth"

        assert ckpt.is_file(), f"Missing checkpoint: {ckpt}"
        assert (out / "run_summary.json").is_file()
        assert not check_run_config(
            out / "run_config.json", common, arms
        )

        # Validation với TTA-D4
        cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
            arms,
            arm,
            str(ckpt),
            DS,
            "val",
            str(BINS),
            str(out / "val_d4"),
            f"{arm}_s{seed}",
            str(arms_path),
            tta="d4"
        )

        subprocess.run(cmd, cwd=PROJECT, check=True)
        print(f"PASS: {arm} seed {seed}", flush=True)

print("\nDONE: B0/B2 seeds 1–4, validation D4")


===== P_B0_LLR | SEED 1 =====
Exp12 configuration check passed: {'arm_id': 'P_B0_LLR', 'common_sha256': 'c28cdcdf035c87b87c8bf25ca787739dd2b34b90868aeed889db4a98010ef2a0', 'arms_sha256': '391914f037d2d5eb515342ab97f6c818b50738cce04b18283fe79e6dfef96737'}
Using cuda
Cuda current device: 0
Cuda device name: Tesla T4

MODEL STATISTICS
Total Parameters: 3,662,291
Trainable Parameters: 3,662,291
Non-trainable Parameters: 0
FLOPs (forward pass): 17,088,643,072 (~17.09G)
FLOPs per pixel: 65,188
Inference time (avg 100 runs): 15.64 ms
Throughput: 63.94 img/s

Logging to: /kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B0_LLR/seed1
Epoch: 0 iter: 1, Loss: 0.96
mIoU: 16.29%
Epoch: 0, EMA Val mIoU: 4.49%
Epoch: 0, Val mIoU: 7.45% | c0=0.13 c1=2.68 c2=19.55
Saved best_val.pth (7.45%)
Epoch: 1, EMA Val mIoU: 20.48%
Epoch: 1, Val mIoU: 30.78% | c0=64.93 c1=10.74 c2=16.69
Saved best_val.pth (30.78%)
Epoch: 2, EMA Val mIoU: 58.49%
Epoch: 2, Val mIoU: 61.79% | c0=95.23 c1=33.11 c2=57.03
Saved best_v

In [33]:
#Bước 1 — Đánh giá B0 seed 0 với D4 để hoàn tất đối chứng,
arm = "P_B0_LLR"
seed = 0

out = PROJECT / "results/exp12" / arm / f"seed{seed}"
ckpt = out / "final_ema.pth"

cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
    arms, arm, str(ckpt), DS, "val",
    str(BINS), str(out / "val_d4"),
    f"{arm}_s{seed}", str(arms_path),
    tta="d4"
)
subprocess.run(cmd, cwd=PROJECT, check=True)

{"run_name": "P_B0_LLR_s0", "split": "val", "prior_override": "none", "prior_noise": "none", "GT_LEAKAGE": false, "instance_source": {"official": 26}, "lambdas": []}
mIoU 84.51 | bg 98.42 crop 77.32 weed 77.80 | BIoU crop 66.43 weed 57.81
  small  n= 620 recall@0.5=56.45 recall@0.3=67.58 cov>=0.5=62.74
  medium n= 567 recall@0.5=75.31 recall@0.3=80.42 cov>=0.5=93.30
  large  n= 621 recall@0.5=50.40 recall@0.3=59.42 cov>=0.5=98.55
  all    n=1808 recall@0.5=60.29 recall@0.3=68.81 cov>=0.5=84.62


CompletedProcess(args=['/usr/bin/python3', 'src/evaluate_ccmc.py', '--dataset_path=/kaggle/input/notebooks/thanhthao28/exp123/weedsgalore-dataset', '--split=val', '--ckpt=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B0_LLR/seed0/final_ema.pth', '--ccmc=False', '--bins_json=/kaggle/working/WeedsGalore_Ipr28/doc/exp02_results/results/exp02/instance_bins.json', '--boundary_px=3', '--run_name=P_B0_LLR_s0', '--out_prefix=/kaggle/working/WeedsGalore_Ipr28/results/exp12/P_B0_LLR/seed0/val_d4', '--arm_id=P_B0_LLR', '--exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_arms.json', '--tta=d4', '--weed_bias=0.0', '--backbone=b0', '--head=bilinear', '--use_low_level_refine=True', '--llr_source=raw', '--input_scale=1.0'], returncode=0)

In [34]:
#Bước 2 — Tính paired CI95 cho B2 so với B0 và TTA-D4 so với không TTA theo đúng compare_runs.py. Chưa chốt B2 chỉ từ trung bình.
# Paired CI95 B0/B2 và TTA-D4
# Cell này đánh giá lại cả 5 seed theo cùng một pipeline, lưu kết quả và tính CI95, không train lại.
import re
import numpy as np
import subprocess
import sys
from scipy.stats import t as student_t
from exp12_config import eval_cli, check_run_config

BINS = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"
assert BINS.is_file()

scores = {}

for arm in ("P_B0_LLR", "P_B2_LLR"):
    for seed in range(5):
        out = PROJECT / "results/exp12" / arm / f"seed{seed}"
        ckpt = out / "final_ema.pth"

        assert ckpt.is_file(), f"Missing: {ckpt}"
        assert not check_run_config(
            out / "run_config.json", common, arms
        )

        for tta in ("none", "d4"):
            cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
                arms, arm, str(ckpt), DS, "val",
                str(BINS), str(out / f"ci_{tta}"),
                f"{arm}_s{seed}",
                str(arms_path),
                tta=tta
            )

            result = subprocess.run(
                cmd, cwd=PROJECT, check=True,
                capture_output=True, text=True
            )

            matches = re.findall(
                r"^mIoU\s+([0-9.]+)\s*\|",
                result.stdout, flags=re.MULTILINE
            )
            assert len(matches) == 1, result.stdout[-3000:]

            scores[(arm, seed, tta)] = float(matches[0])
            print(arm, seed, tta, matches[0], flush=True)


def paired_ci95(name, a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = a - b
    n = len(diff)

    mean = diff.mean()
    sem = diff.std(ddof=1) / np.sqrt(n)
    critical = student_t.ppf(0.975, df=n-1)

    lo = mean - critical * sem
    hi = mean + critical * sem

    print(
        f"{name}: delta={mean:+.3f} pp | "
        f"paired t CI95 [{lo:+.3f}, {hi:+.3f}] | "
        f"{'PASS' if lo > 0 else 'INCONCLUSIVE'}"
    )


def values(arm, tta):
    return [scores[(arm, seed, tta)] for seed in range(5)]


print("\n========== PAIRED CI95 ==========")

paired_ci95(
    "B2 vs B0 | no TTA",
    values("P_B2_LLR", "none"),
    values("P_B0_LLR", "none")
)

paired_ci95(
    "B2 vs B0 | D4",
    values("P_B2_LLR", "d4"),
    values("P_B0_LLR", "d4")
)

for arm in ("P_B0_LLR", "P_B2_LLR"):
    paired_ci95(
        f"{arm} | D4 vs none",
        values(arm, "d4"),
        values(arm, "none")
    )

# Lưu kết quả để không phải đánh giá lại
import json

save_path = PROJECT / "results/exp12/e12_paired_seed_ci95_inputs.json"
save_path.parent.mkdir(parents=True, exist_ok=True)

save_path.write_text(
    json.dumps(
        [
            {
                "arm": arm,
                "seed": seed,
                "tta": tta,
                "val_miou": score
            }
            for (arm, seed, tta), score in scores.items()
        ],
        indent=2
    )
)
print("Saved:", save_path)

P_B0_LLR 0 none 84.02
P_B0_LLR 0 d4 84.51
P_B0_LLR 1 none 83.54
P_B0_LLR 1 d4 83.91
P_B0_LLR 2 none 83.91
P_B0_LLR 2 d4 84.25
P_B0_LLR 3 none 83.75
P_B0_LLR 3 d4 84.01
P_B0_LLR 4 none 83.42
P_B0_LLR 4 d4 83.88
P_B2_LLR 0 none 84.54
P_B2_LLR 0 d4 84.94
P_B2_LLR 1 none 84.44
P_B2_LLR 1 d4 84.87
P_B2_LLR 2 none 84.25
P_B2_LLR 2 d4 84.87
P_B2_LLR 3 none 84.27
P_B2_LLR 3 d4 84.83
P_B2_LLR 4 none 83.55
P_B2_LLR 4 d4 84.10

========== PAIRED CI95 ==========
B2 vs B0 | no TTA: delta=+0.482 pp | paired t CI95 [+0.130, +0.834] | PASS
B2 vs B0 | D4: delta=+0.610 pp | paired t CI95 [+0.242, +0.978] | PASS
P_B0_LLR | D4 vs none: delta=+0.384 pp | paired t CI95 [+0.269, +0.499] | PASS
P_B2_LLR | D4 vs none: delta=+0.512 pp | paired t CI95 [+0.396, +0.628] | PASS
Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp12/e12_paired_seed_ci95_inputs.json


In [38]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "src/scripts/compare_runs.py", "--helpfull"],
    cwd=PROJECT,
    check=False
)

Paired cluster bootstrap between two models (optionally several paired seeds) + gate rule.

Delta = mean_over_seeds(metric(A_seed)) - mean_over_seeds(metric(B_seed)), with the SAME
resampled clusters for every seed and both models (paired by image name). Clusters are
images (default) or tiles (same tile id across flight dates, e.g. '0956').

python src/scripts/compare_runs.py     --a=results/L3_s0_val_per_image.csv,results/L3_s1_val_per_image.csv     --b=results/L2_s0_val_per_image.csv,results/L2_s1_val_per_image.csv     --name_a=L3 --name_b=L2 --gate=h2 --out=results/cmp_L3_vs_L2.json

Gates (plan section 7 / 8.2, thresholds locked BEFORE looking at validation):
  h2 / g3 : SmallWeedRecall delta >= +3.0 pp, 95% CI low > 0, Crop IoU drop <= 1.0 pp
  g4      : SmallWeedRecall delta > 0, 95% CI low > 0, Crop IoU drop <= 1.0 pp
  abs     : L3 - L0, same as h2
  screen  : C3 vs C2 one-way screen (SCREEN-PASS / NO-GO / INCONCLUSIVE), never 'H2 PASS'

flags:

src/scripts/compare_runs.py:
  -

CompletedProcess(args=['/usr/bin/python3', 'src/scripts/compare_runs.py', '--helpfull'], returncode=1)

In [39]:
# Cell chạy compare_runs.py chính thức
# Theo phần hướng dẫn bạn gửi, script hỗ trợ bootstrap ghép cặp theo ảnh và seed. Chạy cell sau:
from pathlib import Path
import subprocess
import sys

ROOT = PROJECT / "results/exp12"
SCRIPT = PROJECT / "src/scripts/compare_runs.py"

assert SCRIPT.is_file()

def csvs(arm, tta):
    paths = []
    for seed in range(5):
        out = ROOT / arm / f"seed{seed}"
        expected = out / f"ci_{tta}_per_image.csv"

        if not expected.is_file():
            matches = sorted(out.glob(f"ci_{tta}*per_image*.csv"))
            assert len(matches) == 1, (
                f"Expected exactly one CSV for {arm} seed{seed} {tta}, "
                f"found: {matches}"
            )
            expected = matches[0]

        paths.append(str(expected))
    return ",".join(paths)


comparisons = [
    ("B2_vs_B0_none", "P_B2_LLR", "none", "P_B0_LLR", "none"),
    ("B2_vs_B0_d4",   "P_B2_LLR", "d4",   "P_B0_LLR", "d4"),
    ("B0_d4_vs_none", "P_B0_LLR", "d4",   "P_B0_LLR", "none"),
    ("B2_d4_vs_none", "P_B2_LLR", "d4",   "P_B2_LLR", "none"),
]

for name, arm_a, tta_a, arm_b, tta_b in comparisons:
    out_json = ROOT / f"compare_{name}.json"

    cmd = [
        sys.executable,
        str(SCRIPT),
        f"--a={csvs(arm_a, tta_a)}",
        f"--b={csvs(arm_b, tta_b)}",
        f"--name_a={arm_a}_{tta_a}",
        f"--name_b={arm_b}_{tta_b}",
        "--gate=none",
        "--primary=mIoU",
        "--metrics=mIoU",
        "--co_primary=",
        "--cluster=image",
        "--seed_bootstrap",
        "--n_boot=10000",
        "--seed=0",
        f"--out={out_json}",
    ]

    print(f"\n===== {name} =====", flush=True)
    subprocess.run(cmd, cwd=PROJECT, check=True)
    print("Saved:", out_json)

print("\nDONE: Hierarchical paired bootstrap")


===== B2_vs_B0_none =====
P_B2_LLR_none vs P_B0_LLR_none | seeds=5 | cluster=image (n=26) | seed_bootstrap=True
metric                    A        B    delta   95% CI           P(d<=0)
mIoU                  84.21    83.73    +0.49   [ +0.22,  +0.79]   0.0003
IoU_crop              76.99    76.01    +0.99   [ +0.40,  +1.63]   0.0002
Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp12/compare_B2_vs_B0_none.json

===== B2_vs_B0_d4 =====
P_B2_LLR_d4 vs P_B0_LLR_d4 | seeds=5 | cluster=image (n=26) | seed_bootstrap=True
metric                    A        B    delta   95% CI           P(d<=0)
mIoU                  84.72    84.11    +0.61   [ +0.28,  +0.96]   0.0
IoU_crop              77.90    76.65    +1.25   [ +0.49,  +2.00]   0.0
Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp12/compare_B2_vs_B0_d4.json

===== B0_d4_vs_none =====
P_B0_LLR_d4 vs P_B0_LLR_none | seeds=5 | cluster=image (n=26) | seed_bootstrap=True
metric                    A        B    delta   95% CI           P(d<=0

In [40]:
# Cell 2 — E12.3: kiểm thử SCI seed 0
# Ưu tiên kiểm thử Raw LLR + Occupancy-Conditioned Sub-cell Decoding. Cell tự kiểm tra arm đã được đăng ký; nếu thiếu sẽ dừng, không tự sửa registry
sci_arms = ["L12_B0_OCC2", "L12_B0_MCSD2"]

for arm in sci_arms:
    assert arm in arms["arms"], f"Arm chưa đăng ký: {arm}"
    assert 0 in arms["arms"][arm]["seeds"], f"Seed 0 chưa đăng ký: {arm}"

for arm in sci_arms:
    print(f"\n===== E12.3 | {arm} | seed 0 =====", flush=True)
    train_one(arm, 0)


===== E12.3 | L12_B0_OCC2 | seed 0 =====
Exp12 configuration check passed: {'arm_id': 'L12_B0_OCC2', 'common_sha256': 'c28cdcdf035c87b87c8bf25ca787739dd2b34b90868aeed889db4a98010ef2a0', 'arms_sha256': '391914f037d2d5eb515342ab97f6c818b50738cce04b18283fe79e6dfef96737'}
Using cuda
Cuda current device: 0
Cuda device name: Tesla T4

MODEL STATISTICS
Total Parameters: 3,663,839
Trainable Parameters: 3,663,839
Non-trainable Parameters: 0
FLOPs (forward pass): 17,189,306,368 (~17.19G)
FLOPs per pixel: 65,572
Inference time (avg 100 runs): 16.12 ms
Throughput: 62.03 img/s

Logging to: /kaggle/working/WeedsGalore_Ipr28/results/exp12/L12_B0_OCC2/seed0
Epoch: 0 iter: 1, Loss: 1.30
mIoU: 16.17%
Epoch: 0, EMA Val mIoU: 0.96%
Epoch: 0, Val mIoU: 1.03% | c0=0.25 c1=2.85 c2=0.00
Saved best_val.pth (1.03%)
Epoch: 1, EMA Val mIoU: 13.61%
Epoch: 1, Val mIoU: 28.48% | c0=42.77 c1=3.64 c2=39.05
Saved best_val.pth (28.48%)
Epoch: 2, EMA Val mIoU: 56.03%
Epoch: 2, Val mIoU: 57.24% | c0=93.43 c1=31.04 c2=47.

In [41]:
#Sau khi chạy xong, đánh giá validation không TTA để so sánh đóng góp kiến trúc trong cùng điều kiện:
# Quyết định: GO cho E12.3 seed 0. Mục tiêu tiếp theo là 
# kiểm tra OccCond/MCSD có cải thiện Small-Weed Recall mà không làm giảm Crop IoU quá mức hay không, trước khi mở rộng lên 5 seed.
from exp12_config import eval_cli
import subprocess
import sys

bins = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"

for arm in ["L12_B0_OCC2", "L12_B0_MCSD2"]:
    seed = 0
    out = PROJECT / "results/exp12" / arm / f"seed{seed}"
    ckpt = out / "final_ema.pth"

    assert ckpt.is_file()

    cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
        arms, arm, str(ckpt), DS, "val",
        str(bins), str(out / "val_none"),
        f"{arm}_s{seed}", str(arms_path),
        tta="none"
    )
    subprocess.run(cmd, cwd=PROJECT, check=True)

{"run_name": "L12_B0_OCC2_s0", "split": "val", "prior_override": "none", "prior_noise": "none", "GT_LEAKAGE": false, "instance_source": {"official": 26}, "lambdas": []}
mIoU 83.51 | bg 98.30 crop 75.74 weed 76.48 | BIoU crop 64.65 weed 55.18
  small  n= 620 recall@0.5=55.48 recall@0.3=69.84 cov>=0.5=65.48
  medium n= 567 recall@0.5=73.90 recall@0.3=79.72 cov>=0.5=93.30
  large  n= 621 recall@0.5=49.92 recall@0.3=57.65 cov>=0.5=98.55
  all    n=1808 recall@0.5=59.35 recall@0.3=68.75 cov>=0.5=85.56
{"run_name": "L12_B0_MCSD2_s0", "split": "val", "prior_override": "none", "prior_noise": "none", "GT_LEAKAGE": false, "instance_source": {"official": 26}, "lambdas": []}
mIoU 83.22 | bg 98.35 crop 74.95 weed 76.38 | BIoU crop 63.98 weed 55.65
  small  n= 620 recall@0.5=53.87 recall@0.3=67.90 cov>=0.5=61.61
  medium n= 567 recall@0.5=73.90 recall@0.3=78.84 cov>=0.5=93.30
  large  n= 621 recall@0.5=50.08 recall@0.3=58.29 cov>=0.5=98.39
  all    n=1808 recall@0.5=58.85 recall@0.3=68.03 cov>=0.5=8

In [ ]:
from pathlib import Path
import subprocess
import zipfile

src = Path("/kaggle/working/WeedsGalore_Ipr28/results/exp12")
dst = Path("/kaggle/working/exp12_backup.zip")

assert src.is_dir()

subprocess.run(
    ["zip", "-q", "-r", str(dst), "exp12"],
    cwd=src.parent,
    check=True
)

with zipfile.ZipFile(dst) as z:
    bad = z.testzip()
    assert bad is None, f"Corrupted file: {bad}"

print("Backup OK:", dst)
print("Size:", round(dst.stat().st_size / 1024**3, 2), "GB")

In [37]:
!cd /kaggle/working/WeedsGalore_Ipr28/results && zip -r /kaggle/working/exp12.zip exp12

updating: exp12/ (stored 0%)
updating: exp12/profile_b2_600_bs1.json (deflated 29%)
updating: exp12/profile_b2_900_bs1.json (deflated 29%)
updating: exp12/profile_b0_900_bs1.json (deflated 29%)
updating: exp12/profile_b0_600_bs1.json (deflated 30%)
updating: exp12/profile_b1_600_bs1.json (deflated 30%)
updating: exp12/profile_b1_900_bs1.json (deflated 30%)
updating: exp12/P_B0_LLR/ (stored 0%)
updating: exp12/P_B0_LLR/seed0/ (stored 0%)
updating: exp12/P_B0_LLR/seed0/best_val.pth (deflated 8%)
updating: exp12/P_B0_LLR/seed0/events.out.tfevents.1794319411.6b4fc48c5829.348.0 (deflated 69%)
updating: exp12/P_B0_LLR/seed0/optimizer.pth (deflated 7%)
updating: exp12/P_B0_LLR/seed0/val_summary.json (deflated 67%)
updating: exp12/P_B0_LLR/seed0/final.pth (deflated 8%)
updating: exp12/P_B0_LLR/seed0/0.pth (deflated 8%)
updating: exp12/P_B0_LLR/seed0/history.jsonl (deflated 76%)
updating: exp12/P_B0_LLR/seed0/run_summary.json (deflated 40%)
updating: exp12/P_B0_LLR/seed0/last.pth (deflated 8%)


In [ ]:
import os
os.chdir('/kaggle/working/WeedsGalore_Ipr28')   # phải đứng ở gốc dự án (script thêm 'src' vào sys.path)
!python src/scripts/e11_fp_vegetation_check.py --dataset_path={DS} --seeds=0,1,2,3,4 --out=results/exp11/fp_veg_check

In [ ]:
!ls results/exp11/fp_veg_check
!zip -qr /kaggle/working/fp_veg_check.zip results/exp11/fp_veg_check

In [ ]:
#E11-00 — Session, commit sạch, khóa tiền đăng ký
from datetime import datetime, timezone
from pathlib import Path
import glob, hashlib, json, os, subprocess, sys

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
assert PROJECT.is_dir(), PROJECT
os.chdir(PROJECT)

dirty = subprocess.check_output(
    ['git', 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
assert not dirty, f'Có thay đổi tracked chưa commit:\n{dirty}'
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = PROJECT / 'results' / 'exp11' / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
for sub in ('logs', 'runs', 'validation', 'diagnostics'):
    (OUT / sub).mkdir()
(PROJECT / 'results' / 'exp11' / 'CURRENT_RUN.txt').write_text(RUN_ID, encoding='utf-8')

if 'DS' not in globals():
    hits = sorted(glob.glob('/kaggle/input/**/2023-05-25/semantics', recursive=True))
    assert hits, 'Hãy Add Input WeedsGalore.'
    DS = str(Path(hits[0]).parent.parent)
assert Path(DS).is_dir(), DS
BINS = PROJECT / 'doc' / 'exp01_results' / 'results' / 'instance_bins.json'
assert BINS.is_file(), BINS
edges = json.loads(BINS.read_text(encoding='utf-8'))['edges']
assert edges == [45.0, 147.0], edges

PREREG = {
    'experiment': 'exp11',
    'git_commit': commit,
    'dataset_path': DS,
    'split_hash_expected': '675667edf3f35f16',
    'evaluation_split': 'val',
    'test_access': False,
    'seeds': [0, 1, 2, 3, 4],
    'target_miou': 86.0,
    'target_scope': 'VALIDATION_ONLY',
    'rho_max': 0.5,
    'component_vote_min_delta_pp': 0.5,
    'activity_min_abs_delta_pp': 0.05,
    'primary_boundary_radius_px': 2,
    'boundary_radii_px': [1, 2, 3],
    'small_weed_area_px': 45,
    'partition_order': ['boundary_r2', 'small_weed', 'component_flip',
                        'background_fp', 'other'],
    'bootstrap': {'n_boot': 10000, 'cluster': 'image', 'seed_bootstrap': True, 'seed': 0},
    'crop_guard_ci_lower_pp': -1.0,
    'sc_llr': {'radius': 1, 'guide_dim': 8, 'embed_dim': 16,
               'alpha_init': 0.1, 'alpha_lr': 0.01, 'occ_weight': 0.0},
}
prereg_path = OUT / 'e11_preregistration.json'
text = json.dumps(PREREG, indent=2, sort_keys=True)
prereg_path.write_text(text, encoding='utf-8')
digest = hashlib.sha256(text.encode('utf-8')).hexdigest()
(OUT / 'e11_preregistration.sha256').write_text(digest, encoding='utf-8')
print('RUN_ID:', RUN_ID, '| commit:', commit)
print('Prereg sha256:', digest)
print('Ngưỡng đã khóa; không sửa prereg sau khi chạy E11-D0.')

In [ ]:
#E11-H — Header khôi phục sau khi restart kernel (chạy đầu mọi phiên mới, không chạy E11-00 lại)
from pathlib import Path
import glob, hashlib, json, os, subprocess, sys

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
os.chdir(PROJECT)
if 'OUT' not in globals():
    RUN_ID = (PROJECT / 'results' / 'exp11' / 'CURRENT_RUN.txt').read_text().strip()
    OUT = PROJECT / 'results' / 'exp11' / RUN_ID
    assert OUT.is_dir(), OUT
PREREG = json.loads((OUT / 'e11_preregistration.json').read_text(encoding='utf-8'))
recorded = (OUT / 'e11_preregistration.sha256').read_text().strip()
import hashlib
current = hashlib.sha256(json.dumps(PREREG, indent=2, sort_keys=True).encode()).hexdigest()
assert current == recorded, 'Prereg đã bị sửa sau khi khóa.'
DS = PREREG['dataset_path']
BINS = PROJECT / 'doc' / 'exp01_results' / 'results' / 'instance_bins.json'
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert head == PREREG['git_commit'], (head, PREREG['git_commit'])
print('OUT:', OUT)

In [ ]:
# E11-P0a — Source contract chẩn đoán và checkpoint Exp10
import glob
import hashlib
import json
from pathlib import Path

EXP10_INPUT = Path(
    '/kaggle/input/notebooks/thanhthao28/exp123/exp10'
)

required = [
    'src/scripts/exp11_error_budget.py',
    'src/scripts/exp11_route_decision.py',
    'tests/test_exp11_error_budget.py',
]

# 1. Source contract
source_hashes = {}

for rel in required:
    path = PROJECT / rel
    assert path.is_file(), f'Thiếu file đã cam kết: {rel}'
    source_hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()

error_budget_src = (
    PROJECT / required[0]
).read_text(encoding='utf-8')

assert 'def aggregate_miou' in error_budget_src
assert "'test'" in error_budget_src or '"test"' in error_budget_src, (
    'Script phải có chốt chặn split test'
)

llr_src = (
    PROJECT / 'src' / 'nets' / 'llr_subcell.py'
).read_text(encoding='utf-8')

assert 'def set_branch_scale' in llr_src, (
    'E10 residual model must expose set_branch_scale'
)

# 2. SHA256 helper
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# 3. Checkpoint discovery
EXP10_MODELS = ('E10B0', 'E10B1', 'E10M1')
ckpts = {}

for model in EXP10_MODELS:
    for seed in PREREG['seeds']:
        run = f'{model}_s{seed}'

        patterns = [
            f'/kaggle/input/**/{run}/best_val.pth',
            str(EXP10_INPUT / '**' / run / 'best_val.pth'),
            str(PROJECT / 'results' / 'exp10' / '**' / run / 'best_val.pth'),
            str(PROJECT / 'runs' / run / 'best_val.pth'),
        ]

        hits = []
        for pattern in patterns:
            hits.extend(glob.glob(pattern, recursive=True))

        hits = sorted(set(hits))

        assert hits, (
            f'Không thấy checkpoint {run}. '
            'Kiểm tra Add Input kết quả Exp10.'
        )

        # All copies must be identical
        hashes = {sha256_file(h) for h in hits}

        assert len(hashes) == 1, (
            f'{run}: nhiều checkpoint khác nhau {hits}'
        )

        # Check split hash when config exists
        cfg = Path(hits[0]).parent / 'run_config.json'

        if cfg.is_file():
            split_hash = json.loads(
                cfg.read_text(encoding='utf-8')
            ).get('split_hash')

            assert split_hash == PREREG['split_hash_expected'], (
                run, split_hash
            )

        # 4. Validation summary discovery
        summary_patterns = [
            f'/kaggle/input/**/{run}_val_summary.json',
            str(EXP10_INPUT / '**' / f'{run}_val_summary.json'),
            str(PROJECT / 'results' / 'exp10' / '**' /
                f'{run}_val_summary.json'),
        ]

        summary_hits = []
        for pattern in summary_patterns:
            summary_hits.extend(
                glob.glob(pattern, recursive=True)
            )

        summary_hits = sorted(set(summary_hits))

        ckpts[run] = {
            'path': hits[0],
            'sha256': next(iter(hashes)),
            'validation_summary': (
                summary_hits[0]
                if len(summary_hits) == 1 else None
            ),
        }

        print(f'CHECKPOINT OK: {run}')

# 5. Contract report
assert len(ckpts) == 15, (
    f'Expected 15 checkpoints, found {len(ckpts)}'
)

report = {
    'status': 'PASS_SOURCE_AND_CKPT_CONTRACT',
    'source_sha256': source_hashes,
    'checkpoints': ckpts,
    'test_accessed': False,
}

(OUT / 'e11_p0a_contract.json').write_text(
    json.dumps(report, indent=2),
    encoding='utf-8'
)

print('=' * 50)
print('E11-P0a PASS:', len(ckpts), 'checkpoints')
print('Report:', OUT / 'e11_p0a_contract.json')


In [ ]:
#E11-P1a — Test chẩn đoán tổng hợp
import subprocess, sys, json
log = OUT / 'logs' / 'e11_p1a_tests.log'
with log.open('w', encoding='utf-8') as f:
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_exp11_error_budget.py'],
                       cwd=PROJECT, stdout=f, stderr=subprocess.STDOUT, text=True)
print(log.read_text(encoding='utf-8')[-3000:])
assert r.returncode == 0, 'Cổng test chẩn đoán thất bại; không chạy D0.'
(OUT / 'e11_p1a_test_summary.json').write_text(
    json.dumps({'status': 'PASS', 'test_data_accessed': False}), encoding='utf-8')

In [ ]:
#E11-D0 — Error budget và activity audit trên 15 checkpoint Exp10 (không train, chỉ validation)
import json, subprocess, sys
assert json.loads((OUT / 'e11_p1a_test_summary.json').read_text())['status'] == 'PASS'
contract = json.loads((OUT / 'e11_p0a_contract.json').read_text(encoding='utf-8'))

BASE = ['--ccmc=False', '--use_low_level_refine=True', '--llr_source=raw', '--spectral_guide=False']
ARCH = {
    'E10B0': ['--head=bilinear'],
    'E10B1': ['--head=llr_residual2', '--beta=0.5', '--residual_alpha_init=0.02'],
    'E10M1': ['--head=llr_residual_occ2', '--beta=0.5', '--residual_alpha_init=0.02'],
}
for run, info in contract['checkpoints'].items():
    model = run.split('_s')[0]
    prefix = OUT / 'diagnostics' / f'{run}'
    if Path(str(prefix) + '_budget.json').is_file():
        print('Đã có:', run); continue
    cmd = [sys.executable, 'src/scripts/exp11_error_budget.py',
           f'--dataset_path={DS}', '--split=val', f'--ckpt={info["path"]}', *BASE, *ARCH[model],
           f'--bins_json={BINS}', f'--run_name={run}', f'--out_prefix={prefix}',
           '--radii=1,2,3', '--primary_radius=2', '--component_vote=True',
           '--activity_audit=True']
    print('Budget:', run)
    subprocess.run(cmd, cwd=PROJECT, check=True)
print('E11-D0 hoàn tất, chạy cổng tái lập tiếp theo.')

In [ ]:
from pathlib import Path
import json

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')

OUT = PROJECT / 'results/exp11/20261108T142939Z'

decision = json.loads(
    (OUT / 'e11_route_decision.json').read_text(encoding='utf-8')
)

assert decision['proceed_to_training'] is True
assert 'A_LOCAL' in decision['eligible_routes']

print('OUT:', OUT)
print('Route:', decision['eligible_routes'])
print('E11-D1 verified: PASS')

In [ ]:
#E11-P0b — Source contract SC-LLR (chỉ chạy nếu A_LOCAL hợp lệ)
import hashlib, json
decision = json.loads((OUT / 'e11_route_decision.json').read_text(encoding='utf-8'))
assert decision['proceed_to_training'] is True and 'A_LOCAL' in decision['eligible_routes'], \
    'Route A không hợp lệ theo ngưỡng đã khóa; không train.'

required = ['src/nets/sc_llr.py', 'tests/test_sc_llr.py', 'src/train.py',
            'src/ccmc_common.py', 'src/evaluate_ccmc.py']
hashes, text = {}, {}
for rel in required:
    path = PROJECT / rel
    assert path.is_file(), f'Thiếu {rel}'
    hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
    text[rel] = path.read_text(encoding='utf-8')
assert 'class SCLLRSegFormer' in text['src/nets/sc_llr.py']
assert "SC_LLR_HEADS = ('sc_llr_static', 'sc_llr_sem', 'sc_llr_spec')" in text['src/nets/sc_llr.py']
for rel in ('src/train.py', 'src/ccmc_common.py'):
    assert 'SC_LLR_HEADS' in text[rel], f'Chưa đăng ký head SC-LLR trong {rel}'
for flag in ('sc_alpha_init', 'sc_alpha_lr', 'sc_radius'):
    assert flag in text['src/train.py'], f'Thiếu cờ {flag}'
assert 'sc_alpha_override' in text['src/evaluate_ccmc.py']
(OUT / 'e11_p0b_contract.json').write_text(
    json.dumps({'status': 'PASS_SC_LLR_CONTRACT', 'source_sha256': hashes}, indent=2),
    encoding='utf-8')
print('E11-P0b PASS')

In [ ]:
#E11-P1b — Test SC-LLR (identity, locality, phụ thuộc phổ/ngữ nghĩa, gradient, tham số)
import json, subprocess, sys
log = OUT / 'logs' / 'e11_p1b_tests.log'
with log.open('w', encoding='utf-8') as f:
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_sc_llr.py'],
                       cwd=PROJECT, stdout=f, stderr=subprocess.STDOUT, text=True)
print(log.read_text(encoding='utf-8')[-3000:])
assert r.returncode == 0, 'Cổng test SC-LLR thất bại; không train, không đổi tên phương pháp để che lỗi.'
(OUT / 'e11_p1b_test_summary.json').write_text(
    json.dumps({'status': 'PASS', 'test_data_accessed': False}), encoding='utf-8')

In [ ]:
#E11-P2 — Train 4 arm × 5 seed (cùng protocol Exp10)
import json, subprocess, sys
assert json.loads((OUT / 'e11_p0b_contract.json').read_text())['status'] == 'PASS_SC_LLR_CONTRACT'
assert json.loads((OUT / 'e11_p1b_test_summary.json').read_text())['status'] == 'PASS'

common = [
    f'--dataset_path={DS}', '--dataset_size_train=104', '--in_channels=5', '--num_classes=3',
    '--batch_size=8', '--num_workers=4', '--lr=0.0003',
    '--weight_bg=0.9', '--weight_crop=1.0', '--weight_weed=1.3',
    '--use_dice_loss=True', '--dice_weight=0.3', '--epochs=100',
    '--log_interval=50', '--ckpt_interval=1000000', '--pretrained_backbone=True',
    '--use_multiscale_fusion=True', '--spectral_guided=False', '--spectral_guide=False',
    '--measure_stats=False', '--resume=True', '--ccmc=False', '--segformer=True',
    '--use_low_level_refine=True', '--llr_source=raw', '--occ_weight=0.0',
]
sc = PREREG['sc_llr']
sc_flags = [f'--sc_radius={sc["radius"]}', f'--sc_guide_dim={sc["guide_dim"]}',
            f'--sc_embed_dim={sc["embed_dim"]}', f'--sc_alpha_init={sc["alpha_init"]}',
            f'--sc_alpha_lr={sc["alpha_lr"]}']
heads = {
    'E11B0': ['--head=bilinear'],
    'E11B1': ['--head=sc_llr_static', *sc_flags],
    'E11M2': ['--head=sc_llr_sem', *sc_flags],
    'E11M1': ['--head=sc_llr_spec', *sc_flags],
}
for seed in PREREG['seeds']:
    for model in ('E11B0', 'E11B1', 'E11M1', 'E11M2'):
        run = f'{model}_s{seed}'
        run_dir = OUT / 'runs' / run
        if (run_dir / 'run_summary.json').is_file():
            print('Đã xong:', run); continue
        log_path = OUT / 'logs' / f'train_{run}.log'
        cmd = [sys.executable, 'src/train.py', *common, *heads[model],
               f'--seed={seed}', f'--run_name={run}', f'--out_dir={run_dir}']
        print('Train:', run)
        with log_path.open('a', encoding='utf-8') as log:
            r = subprocess.run(cmd, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT, text=True)
        if r.returncode != 0:
            print(log_path.read_text(encoding='utf-8')[-4000:])
            raise RuntimeError(f'{run} lỗi; giữ log và dừng.')
        assert (run_dir / 'run_summary.json').is_file(), run
print('E11-P2 xong')

In [ ]:
import json
for model in ('E11B1', 'E11M2', 'E11M1'):
    path = OUT / 'runs' / f'{model}_s0' / 'history.jsonl'
    if not path.is_file():
        continue
    rows = [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]
    a = [r.get('sc_alpha') for r in rows if r.get('sc_alpha') is not None]
    print(model, 'alpha ep0/10/50/cuối:',
          [round(a[min(i, len(a) - 1)], 3) for i in (0, 10, 50, len(a) - 1)] if a else 'KHÔNG GHI')

In [ ]:
#E11-P3 — Đánh giá validation, activity override và so sánh paired
import json, subprocess, sys
from pathlib import Path

BASE = ['--ccmc=False', '--use_low_level_refine=True', '--llr_source=raw',
        '--spectral_guide=False', f'--bins_json={BINS}', '--boundary_px=3']
sc = PREREG['sc_llr']
sc_arch = [f'--sc_radius={sc["radius"]}', f'--sc_guide_dim={sc["guide_dim"]}',
           f'--sc_embed_dim={sc["embed_dim"]}']
archs = {
    'E11B0': ['--head=bilinear'],
    'E11B1': ['--head=sc_llr_static', *sc_arch],
    'E11M2': ['--head=sc_llr_sem', *sc_arch],
    'E11M1': ['--head=sc_llr_spec', *sc_arch],
}
runs = {}

def evaluate(model, seed, tag=None, extra=()):
    run = f'{model}_s{seed}'
    name = f'{tag or model}_s{seed}'
    ckpt = OUT / 'runs' / run / 'best_val.pth'
    assert ckpt.is_file(), ckpt
    prefix = OUT / 'validation' / f'{name}_val'
    cmd = [sys.executable, 'src/evaluate_ccmc.py', f'--dataset_path={DS}', '--split=val',
           f'--ckpt={ckpt}', *BASE, *archs[model], *extra,
           f'--run_name={name}', f'--out_prefix={prefix}']
    subprocess.run(cmd, cwd=PROJECT, check=True)
    assert Path(str(prefix) + '_per_image.csv').is_file()
    runs[name] = str(prefix)

for model in archs:
    for seed in PREREG['seeds']:
        evaluate(model, seed)
for seed in PREREG['seeds']:                       # activity audit: nhánh SC-LLR bị tắt lúc suy luận
    evaluate('E11M1', seed, tag='E11M1zero', extra=['--sc_alpha_override=0.0'])

(OUT / 'e11_p3_validation_manifest.json').write_text(json.dumps(
    {'evaluation_split': 'val', 'test_accessed': False, 'runs': runs}, indent=2), encoding='utf-8')

boot = PREREG['bootstrap']
contrasts = [('E11M1', 'E11B0'), ('E11M1', 'E11B1'), ('E11M1', 'E11M2'),
             ('E11B1', 'E11B0'), ('E11M2', 'E11B0'), ('E11M1', 'E11M1zero')]
for a, b in contrasts:
    pa = [f'{runs[f"{a}_s{s}"]}_per_image.csv' for s in PREREG['seeds']]
    pb = [f'{runs[f"{b}_s{s}"]}_per_image.csv' for s in PREREG['seeds']]
    cmd = [sys.executable, 'src/scripts/compare_runs.py',
           f'--a={",".join(pa)}', f'--b={",".join(pb)}', f'--name_a={a}', f'--name_b={b}',
           f'--cluster={boot["cluster"]}', f'--seed_bootstrap={boot["seed_bootstrap"]}',
           f'--n_boot={boot["n_boot"]}', f'--seed={boot["seed"]}', '--gate=none',
           '--metrics=SmallWeedRecall,cov_small,mIoU,IoU_weed,IoU_crop,BIoU_weed,PrecisionWeed',
           f'--out={OUT / f"cmp_{a}_vs_{b}.json"}']
    subprocess.run(cmd, cwd=PROJECT, check=True)
print('E11-P3 xong; đọc toàn bộ JSON trước khi kết luận.')

In [ ]:
#E11-P4 — GO/NO-GO (ba kết luận tách riêng)
import json

def load(name):
    return json.loads((OUT / f'cmp_{name}.json').read_text(encoding='utf-8'))

def mean_delta(m):
    return m.get('delta', m['A_mean'] - m['B_mean'])

def decision(rec):
    miou, crop = rec['metrics']['mIoU'], rec['metrics']['IoU_crop']
    if miou['ci95'][0] > 0 and crop['ci95'][0] > PREREG['crop_guard_ci_lower_pp']:
        return 'PASS'
    if miou['ci95'][1] <= 0 or crop['ci95'][1] <= PREREG['crop_guard_ci_lower_pp']:
        return 'FAIL'
    return 'INCONCLUSIVE'

m1_b0, m1_b1, m1_m2, m1_zero = (load('E11M1_vs_E11B0'), load('E11M1_vs_E11B1'),
                                load('E11M1_vs_E11M2'), load('E11M1_vs_E11M1zero'))
activity = m1_zero['metrics']['mIoU']
active = abs(mean_delta(activity)) >= PREREG['activity_min_abs_delta_pp']

alpha_final = {}
for model in ('E11B1', 'E11M2', 'E11M1'):
    vals = []
    for seed in PREREG['seeds']:
        p = OUT / 'runs' / f'{model}_s{seed}' / 'history.jsonl'
        rows = [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
        vals.append(rows[-1].get('sc_alpha'))
    alpha_final[model] = vals

spectral_value = 'PASS' if decision(m1_b1) == 'PASS' and decision(m1_m2) == 'PASS' else (
    'FAIL' if m1_b1['metrics']['mIoU']['ci95'][1] <= 0 else 'INCONCLUSIVE')
report = {
    'IMPROVED_VS_LLR': decision(m1_b0),
    'SPECTRAL_GUIDANCE_VALUE': spectral_value,
    'MECHANISM_ACTIVE': 'YES' if active else 'NO_BRANCH_EFFECT_AT_INFERENCE',
    'TARGET_86_REACHED': 'PASS' if m1_b0['metrics']['mIoU']['A_mean'] > PREREG['target_miou'] else 'FAIL',
    'target_scope': 'VALIDATION_ONLY',
    'primary': {'H1_M1_vs_B0': decision(m1_b0), 'H2_M1_vs_B1': decision(m1_b1),
                'H3_M1_vs_M2': decision(m1_m2)},
    'alpha_final_by_arm': alpha_final,
    'test_accessed': False,
    'interpretation': ('Chỉ H1 PASS kèm Crop guard mới hỗ trợ cải thiện đã kiểm chứng so với LLR. '
                       'SPECTRAL_GUIDANCE_VALUE cần cả H2 và H3 PASS. '
                       'Mean validation > 86 không phải claim test.'),
}
(OUT / 'e11_go_no_go.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))

In [ ]:
#E11-P5 — Đóng gói
from zipfile import ZIP_DEFLATED, ZipFile
import glob

res_zip = Path('/kaggle/working') / f'exp11_{PREREG["git_commit"][:7]}_results.zip'
with ZipFile(res_zip, 'w', compression=ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix != '.pth':
            z.write(p, arcname=str(p.relative_to(PROJECT)))
ckpt_zip = Path('/kaggle/working') / f'exp11_{PREREG["git_commit"][:7]}_ckpts.zip'
with ZipFile(ckpt_zip, 'w', compression=ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(str(OUT / 'runs' / '*' / 'best_val.pth'))):
        z.write(p, arcname=str(Path(p).relative_to(PROJECT)))
print(res_zip, ckpt_zip)

In [ ]:
#E11-D0b — Cổng tái lập: mIoU nền phải khớp summary Exp10
import json
from pathlib import Path

rows, status = [], 'PASS'
for run, info in contract['checkpoints'].items():
    budget = json.loads(Path(OUT / 'diagnostics' / f'{run}_budget.json').read_text(encoding='utf-8'))
    summary_path = Path(info['validation_summary']) if info.get('validation_summary') else None
    if summary_path is None or not summary_path.is_file():
        rows.append({'run': run, 'status': 'NO_SUMMARY'}); status = 'PARTIAL'; continue
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    ref = summary.get('metrics', {}).get('mIoU',
                                         summary.get('mIoU', summary.get('miou')))
    assert ref is not None, f'Không thấy khóa mIoU trong {summary_path}; điều chỉnh cell cho đúng schema.'
    diff = abs(float(budget['baseline']['mIoU']) - float(ref))
    ok = diff <= 1e-3
    status = status if ok else 'FAIL'
    rows.append({'run': run, 'budget_mIoU': budget['baseline']['mIoU'], 'summary_mIoU': ref,
                 'abs_diff': diff, 'status': 'PASS' if ok else 'FAIL'})
(OUT / 'e11_d0b_reproduction.json').write_text(
    json.dumps({'status': status, 'rows': rows}, indent=2), encoding='utf-8')
print('Reproduction:', status)
assert status == 'PASS', 'Bộ gộp mIoU hoặc checkpoint không khớp Exp10; dừng trước D1.'

In [ ]:
#E11-D1 — Quyết định route (script đọc ngưỡng đã khóa)
import json, subprocess, sys
assert json.loads((OUT / 'e11_d0b_reproduction.json').read_text())['status'] == 'PASS'
cmd = [sys.executable, 'src/scripts/exp11_route_decision.py',
       f'--budget_dir={OUT / "diagnostics"}',
       f'--prereg={OUT / "e11_preregistration.json"}',
       f'--out={OUT / "e11_route_decision.json"}']
subprocess.run(cmd, cwd=PROJECT, check=True)
decision = json.loads((OUT / 'e11_route_decision.json').read_text(encoding='utf-8'))
for key in ('gap_pp', 'min_ceiling_required_pp', 'ceilings', 'component_vote',
            'activity_audit', 'eligible_routes', 'proceed_to_training', 'reason',
            'E10_INACTIVE_SUSPECTED'):
    print(key, '=', json.dumps(decision.get(key), indent=2))

In [ ]:
!zip -rq /kaggle/working/exp11.zip /kaggle/working/WeedsGalore_Ipr28/results/exp11

In [ ]:
#Cell E1.0 — Biến dùng chung
import os
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp01'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'   # bin cố định cho MỌI thí nghiệm sau này
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
LITE_ARCH = "--ccmc=False --segformer=True --use_low_level_refine=True"
print(COMMON)

In [ ]:
#Cell E1.1 — Bin kích thước cỏ (tam phân vị của train, khóa trước khi xem mô hình)
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} 2>&1 | tee {OUT}/logs/bins.log | head -n 60
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E1.2 — Train L0 seed 0 và seed 1
for seed in (0, 1):
    run = f'L0_s{seed}'
    !python src/train.py {COMMON} {L0_ARCH} --seed={seed} --run_name=L0__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.3 — Train LiteSent seed 0 (cùng pipeline, để làm Diag-D0 trên checkpoint có LLR)
!python src/train.py {COMMON} {LITE_ARCH} --seed=0 --run_name=LiteSent__s0 --out_dir=runs/LiteSent_s0 2>&1 | tee {OUT}/logs/train_LiteSent_s0.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.5 — Diag-D0: recall cỏ nhỏ so với cỏ lớn trong cùng mô hình (bootstrap theo ảnh)


In [ ]:
!pip install thop

In [ ]:
#Cell chạy E1/E2:
!python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --dataset_size_train=104 \
    --segformer=True \
    --spectral_guided=False \
    --use_spectral_indices=False \
    --use_attention=False \
    --dlv3p_do=False \
    --in_channels=5 \
    --num_classes=3 \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --batch_size=8 \
    --num_workers=4 \
    --lr=0.0003 \
    --weight_bg=0.9 \
    --weight_crop=1.0 \
    --weight_weed=1.3 \
    --use_dice_loss=True \
    --dice_weight=0.3 \
    --epochs=100 \
    --seed=0 \
    --out_dir=/kaggle/working/out_dir_segformer_no_refine_seed0 \
    --log_interval=25 \
    --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_segformer_no_refine_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=5 \
    --num_classes=3 \
    --temperature=1.0

In [ ]:
#Cell E5, RGB-only nhiều seed:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=3 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_rgb_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=5 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_5band_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
from pathlib import Path

file = Path("/kaggle/working/WeedsGalore_Ipr28/src/evaluate.py")
text = file.read_text()

old = 'print("Architecture: 5-band SegFormer")'
new = '''print(f"Architecture: SegFormer ({FLAGS.in_channels}-channel, "
              f"LowLevelRefine={FLAGS.use_low_level_refine})")'''

assert old in text, "Không tìm thấy dòng cũ hoặc file đã thay đổi!"

file.write_text(text.replace(old, new, 1))
print("Đã cập nhật thành công dòng 65!")

In [ ]:
%cd /kaggle/working/WeedsGalore_Ipr28

import subprocess
import sys

cmd = [
    sys.executable,
    "src/calibrate_temperature.py",
    "--dataset_path", "/kaggle/working/weedsgalore-dataset",
    "--ckpt", "/kaggle/working/out_dir_rgb_seed0/best_val.pth",
    "--segformer=True",
    "--pretrained_backbone=True",
    "--spectral_guided=False",
    "--in_channels", "3",
    "--num_classes", "3",
    "--use_low_level_refine=True",
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("Exit code:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)
print("\n===== STDERR =====")
print(result.stderr)

In [ ]:
import subprocess
import sys
import re
import numpy as np
from pathlib import Path

REPO = "/kaggle/working/WeedsGalore_Ipr28"
DATASET = "/kaggle/working/weedsgalore-dataset"

# Trong mọi subprocess, chạy từ thư mục repository
def run_command(command):
    result = subprocess.run(
        command,
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout + result.stderr

experiments = {
    "RGB": [
        f"/kaggle/working/out_dir_rgb_seed{i}/best_val.pth"
        for i in range(5)
    ],
    "5-band": [
        f"/kaggle/working/out_dir_5band_seed{i}/best_val.pth"
        for i in range(5)
    ],
}

results = {}

for name, checkpoints in experiments.items():
    miou_values = []
    ece_values = []
    temperatures = []

    in_channels = 3 if name == "RGB" else 5

    for seed, checkpoint in enumerate(checkpoints):
        if not Path(checkpoint).exists():
            print(f"SKIP: {checkpoint}")
            continue

        common = [
            f"--dataset_path={DATASET}",
            f"--ckpt={checkpoint}",
            "--segformer=True",
            "--pretrained_backbone=True",
            "--spectral_guided=False",
            f"--in_channels={in_channels}",
            "--num_classes=3",
            "--use_low_level_refine=True",
        ]

        calibration_log = run_command(
            [sys.executable, "src/calibrate_temperature.py"] + common
        )

        match = re.search(
            r"Fitted temperature:\s*T\s*=\s*([0-9.]+)",
            calibration_log,
        )
        if match is None:
            raise RuntimeError(
                f"Không đọc được temperature cho {name}, seed {seed}"
            )

        temperature = float(match.group(1))
        temperatures.append(temperature)

        evaluation_log = run_command([
            sys.executable,
            "src/evaluate.py",
            "--split=test",
            *common,
            f"--temperature={temperature}",
        ])

        print(evaluation_log)

        miou_match = re.search(r"mIoU:\s*([0-9.]+)%", evaluation_log)
        ece_match = re.search(r"ECE:\s*(?:tensor\()?([0-9.]+)", evaluation_log)

        if miou_match is None or ece_match is None:
            raise RuntimeError(f"Không đọc được kết quả cho {name}, seed {seed}")

        miou_values.append(float(miou_match.group(1)))
        ece_values.append(float(ece_match.group(1)))

    results[name] = {
        "mIoU": miou_values,
        "ECE": ece_values,
        "T": temperatures,
    }

print("\n===== SUMMARY =====")
for name, values in results.items():
    print(f"\n{name}")
    print("Temperatures:", values["T"])
    print(
        f"Test mIoU: {np.mean(values['mIoU']):.2f} "
        f"+/- {np.std(values['mIoU'], ddof=1):.2f}%"
    )
    print(
        f"Test ECE: {np.mean(values['ECE']):.5f} "
        f"+/- {np.std(values['ECE'], ddof=1):.5f}"
    )